# Notebook 3: Star Schema Dimensional Modeling

## 1. Architectural Overview & Design Rationale
To scale ITB's business intelligence (BI) capabilities, we must transition from processing a flat, denormalized master table to establishing a highly optimized **Star Schema Data Warehouse** model.

### 1.1. Why Star Schema?
*   **Query Performance:** Minimizes complex SQL table joins, allowing downstream visualization tools (such as Power BI and Tableau) to execute filter queries at maximum efficiency.
*   **Single Source of Truth:** Strictly separates qualitative descriptive attributes (Dimensions) from quantitative, measurable business metrics (Facts).
*   **User Usability:** Creates an intuitive, self-explanatory drag-and-drop model for business users and non-technical stakeholders.

### 1.2. Architecture Design
Our central schema consists of:
*   **`fact_order` (Central Fact Table):** Contains transactional measures (`product_cost`, `product_price`) and surrogate keys pointing to dimensions.
*   **Dimension Tables (Dimensions):**
    *   `dim_customer`: Unified demographic, education, and occupation master.
    *   `dim_product`: Categorical product catalog.
    *   `dim_stores`: Physical store geographic directory.
    *   `dim_date`: Temporal dimension supporting calendars, quarters, and days.
    *   `dim_order_status`: Order status map.
    *   `dim_channel`: Selling channel directory.

In [1]:
import pandas as pd
import numpy as np
import os

# Path resolution to load the processed master file
processed_data_path = '../data/processed/master_df.csv'
if not os.path.exists(processed_data_path):
    processed_data_path = 'data/processed/master_df.csv'

# Load the Consolidated Master Data
try:
    df_master = pd.read_csv(processed_data_path, low_memory=False)
    # Convert dates to datetime type
    df_master['order_date'] = pd.to_datetime(df_master['order_date'])
    df_master['dob'] = pd.to_datetime(df_master['dob'])
    print(f"Successfully loaded Consolidated Master Data. Shape: {df_master.shape}")
except FileNotFoundError as e:
    print(f"Error: {e}. Please run previous notebooks first.")


print("\n=== 1. Generating dim_product ===")
product_cols = ['product_code', 'product_profile', 'goods_category', 'category_product_code']
dim_product = df_master[product_cols].drop_duplicates(subset=['product_code']).copy().reset_index(drop=True)
# Generate surrogate key for Product dimension
dim_product.insert(0, 'product_key', range(1, 1 + len(dim_product)))
print(f"dim_product successfully modeled. Shape: {dim_product.shape}")


print("\n=== 2. Generating dim_customer ===")
customer_cols = ['customer_id', 'name_full', 'gender', 'address', 'region', 'dob', 'age', 'education_level', 'job_title']
dim_customer = df_master[customer_cols].drop_duplicates(subset=['customer_id']).copy().reset_index(drop=True)
# Generate surrogate key for Customer dimension
dim_customer.insert(0, 'customer_key', range(1, 1 + len(dim_customer)))
print(f"dim_customer successfully modeled. Shape: {dim_customer.shape}")

Successfully loaded Consolidated Master Data. Shape: (2049263, 24)

=== 1. Generating dim_product ===
dim_product successfully modeled. Shape: (3132, 5)

=== 2. Generating dim_customer ===
dim_customer successfully modeled. Shape: (849472, 10)


## 2. Generating Remaining Dimensions & Building Central Fact Table (fact_order)

With the product and customer dimensions successfully established, we proceed to:
1.  **Model remaining dimensions:**
    *   `dim_stores`: Contains unique physical retail stores. Placeholder store codes (negative numbers representing online channels) are excluded to keep the physical dimension clean.
    *   `dim_date`: Extracted from unique transaction dates to provide granular attributes (Calendar Date, Year, Quarter, Month, Day, and Weekday name) with an integer surrogate key (`YYYYMMDD`).
    *   `dim_order_status`: Standard status descriptions with unique integer keys.
    *   `dim_channel`: Standard selling channels with unique integer keys.
2.  **Assemble `fact_order`:** Map raw transactions to dimension surrogate keys. Transaction metrics (`product_cost`, `product_price`) are retained at the granular order level.
3.  **Export Star Schema:** Save all dimension and fact tables to `data/processed/` as clean CSVs, fully prepped for Power BI ingestion.

In [3]:
print("=== 3. Generating dim_stores ===")
# Isolate physical stores (filtering out negative placeholder codes of ecommerce/website)
df_physical_stores = df_master[df_master['store_code'] > 0].copy()
store_cols = ['store_code', 'store_region', 'store_district']
dim_stores = df_physical_stores[store_cols].drop_duplicates(subset=['store_code']).copy().reset_index(drop=True)
dim_stores.insert(0, 'store_key', range(1, 1 + len(dim_stores)))
print(f"dim_stores modeled. Shape: {dim_stores.shape}")


print("\n=== 4. Generating dim_date ===")
# Extract unique order dates across the entire dataset
unique_dates = df_master['order_date'].dt.floor('D').unique()
dim_date = pd.DataFrame({'fully_date': unique_dates}).sort_values(by='fully_date').reset_index(drop=True)
dim_date['date_key'] = dim_date['fully_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['fully_date'].dt.year
dim_date['quarter'] = dim_date['fully_date'].dt.quarter
dim_date['month'] = dim_date['fully_date'].dt.month
dim_date['day'] = dim_date['fully_date'].dt.day
dim_date['weekday'] = dim_date['fully_date'].dt.day_name()
dim_date = dim_date[['date_key', 'fully_date', 'year', 'quarter', 'month', 'day', 'weekday']]
print(f"dim_date modeled. Shape: {dim_date.shape}")


print("\n=== 5. Generating dim_order_status ===")
unique_statuses = df_master['order_status'].unique()
dim_order_status = pd.DataFrame({'status_description': unique_statuses})
dim_order_status.insert(0, 'order_status_key', range(1, 1 + len(dim_order_status)))
print(f"dim_order_status modeled. Shape: {dim_order_status.shape}")


print("\n=== 6. Generating dim_channel ===")
unique_channels = df_master['sales_channel'].unique()
dim_channel = pd.DataFrame({'channel_name': unique_channels})
dim_channel.insert(0, 'channel_key', range(1, 1 + len(dim_channel)))
print(f"dim_channel modeled. Shape: {dim_channel.shape}")


print("\n=== 7. Assembling Central FACT_ORDER Table ===")
df_fact = df_master.copy()

# Crucial fix: Drop pre-existing customer_key from transactional file to prevent merge suffixes (_x, _y)
if 'customer_key' in df_fact.columns:
    df_fact.drop(columns=['customer_key'], inplace=True)

# A. Map CustomerKey
df_fact = pd.merge(df_fact, dim_customer[['customer_id', 'customer_key']], on='customer_id', how='left')

# B. Map ProductKey
df_fact = pd.merge(df_fact, dim_product[['product_code', 'product_key']], on='product_code', how='left')

# C. Map StoreKey (online transactions will gracefully retain a null/NaN store_key)
df_fact = pd.merge(df_fact, dim_stores[['store_code', 'store_key']], on='store_code', how='left')
df_fact['store_key'] = df_fact['store_key'].astype('Int64') # Nullable integer type for online sales

# D. Map DateKey
df_fact['date_key'] = df_fact['order_date'].dt.strftime('%Y%m%d').astype(int)

# E. Map OrderStatusKey
df_fact = pd.merge(df_fact, dim_order_status, left_on='order_status', right_on='status_description', how='left')

# F. Map ChannelKey
df_fact = pd.merge(df_fact, dim_channel, left_on='sales_channel', right_on='channel_name', how='left')

# G. Select final fact columns
fact_columns = [
    'order_id', 'customer_key', 'product_key', 'store_key', 
    'date_key', 'order_status_key', 'channel_key', 'product_cost', 'product_price'
]
fact_order = df_fact[fact_columns].copy()

print(f"fact_order successfully modeled. Shape: {fact_order.shape}")
print(f"Verify fact row count integrity: {len(fact_order) == len(df_master)}")


print("\n=== 8. Exporting Dimensional Schema for BI Ingestion ===")
# Define destination path
processed_dir = '../data/processed/'
if not os.path.exists(processed_dir):
    processed_dir = 'data/processed/'

# Export dimensions and fact table
dim_customer.to_csv(os.path.join(processed_dir, 'dim_customer.csv'), index=False, encoding='utf-8-sig')
dim_product.to_csv(os.path.join(processed_dir, 'dim_product.csv'), index=False, encoding='utf-8-sig')
dim_stores.to_csv(os.path.join(processed_dir, 'dim_stores.csv'), index=False, encoding='utf-8-sig')
dim_date.to_csv(os.path.join(processed_dir, 'dim_date.csv'), index=False, encoding='utf-8-sig')
dim_order_status.to_csv(os.path.join(processed_dir, 'dim_order_status.csv'), index=False, encoding='utf-8-sig')
dim_channel.to_csv(os.path.join(processed_dir, 'dim_channel.csv'), index=False, encoding='utf-8-sig')
fact_order.to_csv(os.path.join(processed_dir, 'fact_order.csv'), index=False, encoding='utf-8-sig')

print(f"All 6 Dimensions and 1 Fact table exported successfully to '{processed_dir}'!")

=== 3. Generating dim_stores ===
dim_stores modeled. Shape: (17467, 4)

=== 4. Generating dim_date ===
dim_date modeled. Shape: (1095, 7)

=== 5. Generating dim_order_status ===
dim_order_status modeled. Shape: (5, 2)

=== 6. Generating dim_channel ===
dim_channel modeled. Shape: (4, 2)

=== 7. Assembling Central FACT_ORDER Table ===
fact_order successfully modeled. Shape: (2049263, 9)
Verify fact row count integrity: True

=== 8. Exporting Dimensional Schema for BI Ingestion ===
All 6 Dimensions and 1 Fact table exported successfully to '../data/processed/'!
